## Local Inference on GPU
Model page: https://huggingface.co/cointegrated/rubert-tiny2

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/cointegrated/rubert-tiny2)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
!pip install transformers datasets seqeval accelerate evaluate -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
STA NER fine-tuning script for Google Colab.
Base model: cointegrated/rubert-tiny2
Task: token classification (NER) for meal planning voice commands.

Usage:
  1. Upload your training JSON files to Colab (or mount Google Drive).
  2. Run all cells top to bottom.
  3. Fine-tuned model is saved to /content/sta-ner-model (and optionally Google Drive).

Training data format (from sta-ner-training-data-prompt.md):
  [
    {
      "text": "добавь курицу 150 грамм на ужин",
      "intent": "add_dish",
      "tokens": ["добавь", "курицу", "150", "грамм", "на", "ужин"],
      "labels": ["O", "B-FOOD_NAME", "B-QUANTITY", "B-UNIT", "O", "B-MEAL_TYPE"]
    },
    ...
  ]
"""

# ── 0. Install dependencies (run once in Colab) ───────────────────────────────
# !pip install transformers datasets seqeval accelerate -q

import json
import glob
import numpy as np
import torch
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from datasets import Dataset
import evaluate

# ── 1. Config ─────────────────────────────────────────────────────────────────

MODEL_NAME = "cointegrated/rubert-tiny2"
TRAINING_DATA_DIR = "/content/drive/MyDrive/MealAhead/train_data_text_to_meal_v2"   # folder with your JSON files
OUTPUT_DIR = "/content/sta-ner-model"
DRIVE_OUTPUT = "/content/drive/MyDrive/MealAhead/train_data_text_to_meal_v2/output"  # set None to skip Drive save

MAX_LENGTH = 128
EPOCHS = 5
BATCH_SIZE = 16
LEARNING_RATE = 3e-5

# ── 2. Label map ──────────────────────────────────────────────────────────────
# INTENT is sentence-level (top-level field in JSON) — not labeled at token level.

LABELS = [
    "O",
    "B-MEAL_TITLE",  "I-MEAL_TITLE",
    "B-MEAL_TYPE",   "I-MEAL_TYPE",
    "B-FOOD_NAME",   "I-FOOD_NAME",
    "B-QUANTITY",    "I-QUANTITY",
    "B-UNIT",        "I-UNIT",
    "B-DATE",        "I-DATE",
    "B-TIME",        "I-TIME",
]

LABEL2ID = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}


In [ ]:
# ── 3. Load training data ─────────────────────────────────────────────────────

def load_samples(data_dir: str) -> list[dict]:
    samples = []
    for path in glob.glob(f"{data_dir}/*.json"):
        with open(path) as f:
            samples.extend(json.load(f))
    print(f"Loaded {len(samples)} samples from {data_dir}")
    return samples

# ── 4. Tokenize with label alignment ─────────────────────────────────────────
# RuBERT uses WordPiece — words split into subwords.
# Only the first subword of each word gets the real label; others get -100 (ignored in loss).

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(samples: list[dict]) -> Dataset:
    all_input_ids, all_attention_masks, all_labels = [], [], []

    for sample in samples:
        word_tokens = sample["tokens"]
        word_labels = [LABEL2ID.get(l, 0) for l in sample["labels"]]

        encoding = tokenizer(
            word_tokens,
            is_split_into_words=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
        )

        label_ids = []
        previous_word_idx = None
        for word_idx in encoding.word_ids():
            if word_idx is None:
                label_ids.append(-100)          # [CLS], [SEP], [PAD]
            elif word_idx != previous_word_idx:
                label_ids.append(word_labels[word_idx])  # first subword → real label
            else:
                label_ids.append(-100)          # continuation subword → ignore
            previous_word_idx = word_idx

        all_input_ids.append(encoding["input_ids"])
        all_attention_masks.append(encoding["attention_mask"])
        all_labels.append(label_ids)

    return Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    })

# ── 5. Metrics ────────────────────────────────────────────────────────────────

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        true_labels.append([ID2LABEL[l] for l in label_row if l != -100])
        true_preds.append([ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100])

    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": result["overall_precision"],
        "recall": result["overall_recall"],
        "f1": result["overall_f1"],
        "accuracy": result["overall_accuracy"],
    }




In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────

def train():
    samples = load_samples(TRAINING_DATA_DIR)

    # 90/10 train/eval split
    split = int(len(samples) * 0.9)
    train_dataset = tokenize_and_align(samples[:split])
    eval_dataset  = tokenize_and_align(samples[split:])

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=10,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Model saved to {OUTPUT_DIR}")

    # Optionally save to Google Drive
    if DRIVE_OUTPUT:
        import shutil
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
        print(f"Model copied to Google Drive: {DRIVE_OUTPUT}")

In [ ]:
train()

Loaded 6454 samples from /content/drive/MyDrive/MealAhead/train_data_text_to_meal_v2


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params we

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.169280,0.197048,0.873947,0.932005,0.902042,0.956150
2,0.050341,0.100289,0.947388,0.971082,0.959089,0.979537
3,0.023321,0.093186,0.953746,0.974990,0.964251,0.982235
4,0.022351,0.094790,0.954111,0.974990,0.964438,0.983360
5,0.016787,0.085649,0.956755,0.976944,0.966744,0.984259


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/sta-ner-model
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model copied to Google Drive: /content/drive/MyDrive/MealAhead/train_data_text_to_meal_v2/output
